# Faruq-v3 — DSRDet/FBNR Breadth Screening (Seed 42)

Training-only foreground/background regularization screening inspired by Xu et al., Pattern Recognition 2026. Four arms are frozen before validation: **BGL1** spatial linear background control, **BGG1** gradient-domain BRBB, **FGC1** coffee-adapted Gaussian foreground concealment, and **FBR1** stochastic decoupled combination. The aircraft-specific cross-shape and instance-rotation prior are deliberately not claimed as transferred. Test remains unavailable and locked.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/dsr-fbnr-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO}[dev]'], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('REPO  :', REPO)
print('BRANCH:', BRANCH)


In [ ]:
tests = ['tests/test_fbnr_transfer.py']
subprocess.run([sys.executable,'-m','pytest','-q',*tests], cwd=REPO, check=True)
from coffee_detector.experiments.run_faruq_v3_fbnr_screening import CONFIGS
print('ARMS:', list(CONFIGS))
assert set(CONFIGS) == {'BGL1','BGG1','FGC1','FBR1'}
print('LOCAL COLAB PRE-FLIGHT: PASS')


## Transfer boundary

- **BGL1**: linear spatial blending control, matching the paper's comparison baseline rather than the proposed BRBB.
- **BGG1**: Sobel gradient extraction, stronger-gradient selection, divergence, FFT Poisson reconstruction, and source-foreground preservation.
- **FGC1**: Gaussian soft concealment and paper-selected dynamic radius `[0.5, 0.8]`; coffee adaptation replaces the oriented-aircraft cross prior with horizontal/vertical bbox axes and omits aircraft-specific instance rotation.
- **FBR1**: original/foreground/background regularization sampled per image at fixed training-update budget. This is a compute-controlled approximation to the paper's three-parallel-input training, not a literal reproduction.

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan GPU Colab.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
D0_CHECKPOINT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
CONTROL = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-dsrdet-fbnr-screening-v2'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('GPU     :', torch.cuda.get_device_name(0))
print('CONTROL :', CONTROL)
print('OUTPUT  :', OUTPUT_ROOT)
for code in ('BGL1','BGG1','FGC1','FBR1'):
    last = OUTPUT_ROOT / f'{code}_seed42/weights/last.pt'
    best = OUTPUT_ROOT / f'{code}_seed42/weights/best.pt'
    print(f'{code:5s}:', 'COMPLETE' if best.is_file() else ('RESUME' if last.is_file() else 'START'))


In [ ]:
command = [
    sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_fbnr_screening',
    '--data-root',str(DATA_ROOT),
    '--grouped-summary',str(GROUPED_SUMMARY),
    '--control-summary',str(CONTROL),
    '--d0-checkpoint',str(D0_CHECKPOINT),
    '--output-root',str(OUTPUT_ROOT),
    '--seed','42','--device','0','--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'FBNR screening gagal, return code={return_code}; traceback lengkap tercetak di atas.')


In [ ]:
import json, pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'val_reports/fbnr_seed42_screening.json'
assert SUMMARY.is_file(), f'Screening belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
assert result['test_opened'] is False
metrics = ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')
rows = []
for model, values in result['controls'].items(): rows.append({'model':model, **values, 'decision':'CONTROL'})
for model, payload in result['candidates'].items(): rows.append({'model':model, **payload['metrics'], 'decision':result['decisions'][model]['decision']})
display(pd.DataFrame(rows).style.format({name:'{:.2%}' for name in metrics}))
print('MECHANISTIC COMPARISONS')
for name, delta in result['mechanistic_comparisons'].items(): print(name, delta)
print('TRANSFER BOUNDARY:', result['paper_transfer_boundary'])
print('SUMMARY:', SUMMARY)
print('Test tetap terkunci. RETAIN di sini hanya discovery signal; bukan klaim final dan bukan izin membuka test.')
